# 01 — Data Import & Exploration

**Goal:** Load raw PPG and blood pressure recordings for all patients, handle timestamp resets in continuous recordings, remove corrupted sensor values, and visualize the PLETH signal distribution.

**Inputs:** `data/raw/train/BP_Table_<id>.pkl` and `PPG_Table_<id>.pkl` for each patient

**Outputs saved to `data/processed/`:**
- `bp_data.pkl` — dict of {patient_id: DataFrame}
- `ppg_clean.pkl` — dict of {patient_id: DataFrame} with saturation values replaced by NaN
- `ppg_segments.pkl` — dict of {patient_id: list[DataFrame]}, split at timestamp resets

---
### Pipeline position
```
[01 Import] → 02 Signal Processing → 03 Interpolation → 04 ML Data Loading → 05 Features → 06 Models
```

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Paths are resolved relative to the repo root regardless of where the
# notebook is opened from.
REPO_ROOT     = Path.cwd().parent if Path.cwd().name == 'improved' else Path.cwd().parent.parent
RAW_TRAIN_DIR = REPO_ROOT / 'data' / 'raw' / 'train'
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# PLETH values produced by sensor saturation or hardware faults.
# These are not real physiological measurements and must be removed.
SATURATION_VALUES = {1, 3276, 62258, 65535}

print(f'Repo root      : {REPO_ROOT}')
print(f'Raw train data : {RAW_TRAIN_DIR}')
print(f'Processed dir  : {PROCESSED_DIR}')

## 1. Load Raw Data

Each patient has two `.pkl` files:
- `BP_Table_<id>.pkl` — invasive blood pressure measurements (arterial line)
- `PPG_Table_<id>.pkl` — photoplethysmography signal from a pulse oximeter

In [ ]:
def extract_patient_id(filename: str) -> str:
    """Return the numeric patient ID from a filename like 'BP_Table_12345678.pkl'."""
    return Path(filename).stem.split('_')[-1]


def load_pickle_files(folder: Path, prefix: str) -> dict:
    """
    Load all .pkl files in *folder* whose name starts with *prefix*.

    Returns a dict mapping patient_id (str) -> DataFrame, with 'patient_id'
    inserted as the first column if not already present.
    """
    data = {}
    for path in sorted(folder.glob(f'{prefix}_*.pkl')):
        pid = extract_patient_id(path.name)
        with open(path, 'rb') as f:
            df = pickle.load(f)
        if 'patient_id' not in df.columns:
            df.insert(0, 'patient_id', pid)
        data[pid] = df
    return data


bp_data  = load_pickle_files(RAW_TRAIN_DIR, 'BP_Table')
ppg_data = load_pickle_files(RAW_TRAIN_DIR, 'PPG_Table')

print(f'Loaded {len(bp_data):>3} blood pressure recordings')
print(f'Loaded {len(ppg_data):>3} PPG recordings')

## 2. Handle Timestamp Resets

The monitoring device occasionally restarts mid-recording, causing the `TIME` column to jump back to zero. Each continuous segment must be treated independently to avoid creating phantom signals at the boundaries.

In [ ]:
def split_at_resets(df: pd.DataFrame, time_col: str = 'TIME') -> list:
    """
    Split *df* wherever the time column decreases (recording restart).

    Returns a list of DataFrames, each representing one continuous segment.
    If *time_col* is not in the DataFrame, the original DataFrame is returned
    as a single-element list.
    """
    if time_col not in df.columns:
        return [df]
    reset_mask = df[time_col].diff() < 0
    break_indices = df.index[reset_mask].tolist()
    cut_points = [0] + break_indices + [len(df)]
    return [
        df.iloc[cut_points[i]: cut_points[i + 1]].reset_index(drop=True)
        for i in range(len(cut_points) - 1)
    ]


ppg_segments = {pid: split_at_resets(df) for pid, df in ppg_data.items()}
segment_counts = {pid: len(segs) for pid, segs in ppg_segments.items()}

print(f'Total segments after timestamp-reset split : {sum(segment_counts.values())}')
print(f'Patients with multiple segments            : {sum(v > 1 for v in segment_counts.values())}')

## 3. Remove Saturation / Corrupted Values

Known bad PLETH values (hardware saturation codes: 1, 3276, 62258, 65535) and zeros are replaced with `NaN` so that downstream processing can detect and handle gaps explicitly.

In [ ]:
def filter_saturation_values(df: pd.DataFrame, pleth_col: str = 'PLETH') -> pd.DataFrame:
    """
    Replace known sensor saturation codes and zeros with NaN.

    These values indicate hardware clipping or signal loss, not real
    physiological measurements.
    """
    df = df.copy()
    df[pleth_col] = df[pleth_col].replace(list(SATURATION_VALUES), np.nan)
    df.loc[df[pleth_col] == 0, pleth_col] = np.nan
    return df


ppg_clean = {pid: filter_saturation_values(df) for pid, df in ppg_data.items()}

total_samples   = sum(len(df) for df in ppg_data.values())
removed_samples = sum(df['PLETH'].isna().sum() for df in ppg_clean.values())
print(f'Total PLETH samples  : {total_samples:>10,}')
print(f'Saturation / zero    : {removed_samples:>10,}  ({100 * removed_samples / total_samples:.1f}%)')
print(f'Usable samples       : {total_samples - removed_samples:>10,}')

## 4. Signal Distribution Analysis

Visualise the PLETH value distribution across all patients to understand the signal range and confirm that saturation values have been removed.

In [ ]:
all_pleth = pd.concat(
    [df['PLETH'].dropna() for df in ppg_clean.values()],
    ignore_index=True,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Raw count histogram
axes[0].hist(all_pleth, bins=200, color='steelblue', edgecolor='none', alpha=0.85)
axes[0].set_title('PLETH value distribution (all patients, cleaned)')
axes[0].set_xlabel('PLETH value')
axes[0].set_ylabel('Count')

# Normalised frequency per patient (bar chart, top-20 most frequent values)
top_vals = all_pleth.value_counts().nlargest(30)
norm_vals = top_vals / top_vals.sum()
axes[1].bar(norm_vals.index.astype(str), norm_vals.values, color='steelblue', alpha=0.85)
axes[1].set_title('Top-30 PLETH values — normalised frequency')
axes[1].set_xlabel('PLETH value')
axes[1].set_ylabel('Normalised frequency')
axes[1].tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.show()

print(f'Mean : {all_pleth.mean():.1f}   Std : {all_pleth.std():.1f}')
print(f'Min  : {all_pleth.min():.0f}    Max : {all_pleth.max():.0f}')

## 5. Save Processed Data

In [ ]:
for filename, obj in [
    ('bp_data.pkl',      bp_data),
    ('ppg_clean.pkl',    ppg_clean),
    ('ppg_segments.pkl', ppg_segments),
]:
    with open(PROCESSED_DIR / filename, 'wb') as f:
        pickle.dump(obj, f)
    print(f'Saved → data/processed/{filename}')